In [1]:
# Install the UCI repository package
!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Fetch dataset
credit = fetch_ucirepo(id=144)
X = credit.data.features.copy()
y = credit.data.targets.iloc[:, 0].copy()

# Map target labels: 1 (Good Risk) -> 0, 2 (Bad Risk) -> 1
y = y.map({1: 0, 2: 1})

print("--- Dataset Dimensions ---")
print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")


--- Dataset Dimensions ---
Features shape: (1000, 20)
Target distribution:
class
0    700
1    300
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, classification_report

# 1. Split into training (80%) and testing (20%) sets using stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2. Automatically group column types
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

# 3. Create preprocessing transformers
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

# 4. Construct the modeling pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 5. Train the model
model.fit(X_train, y_train)

# 6. Evaluate and print model performance
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("===== Credit Risk Model Evaluation =====")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score:   {recall_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score:  {roc_auc_score(y_test, y_proba):.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# 7. Extract feature importance coefficients
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})
importance['AbsoluteImportance'] = importance['Coefficient'].abs()
importance = importance.sort_values('AbsoluteImportance', ascending=False)

print("\n===== Top 15 Most Important Features =====")
print(importance.head(15))


===== Credit Risk Model Evaluation =====
Accuracy Score: 0.7800
Recall Score:   0.5333
ROC-AUC Score:  0.8040

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       140
           1       0.67      0.53      0.59        60

    accuracy                           0.78       200
   macro avg       0.74      0.71      0.72       200
weighted avg       0.77      0.78      0.77       200


===== Top 15 Most Important Features =====
                  Feature  Coefficient  AbsoluteImportance
23    cat__Attribute4_A46     0.901684            0.901684
10    cat__Attribute1_A14    -0.887112            0.887112
15    cat__Attribute3_A34    -0.800579            0.800579
17    cat__Attribute4_A41    -0.714820            0.714820
46  cat__Attribute12_A124     0.713710            0.713710
7     cat__Attribute1_A11     0.657834            0.657834
26    cat__Attribute6_A61     0.640212            0.640212
29    cat__Att